# CurvLearn — Per-subspace mixed curvature (product manifold)

Design B showed the model learns a single useful curvature in the crossover regime. This asks
the next question from the original vision: let **each attention head choose its own
curvature** — a product manifold ⊕_h M_{κ_h} with one learnable κ per head.

Compare, in the crossover regime (gyro, medium@d=4/h=2, hard@d=8/h=4):

- **shared** — one learnable κ for all heads (control).
- **per_head** — a distinct learnable κ per head (mixed curvature).

Two reads: does per_head lower close-loss vs shared, and do the heads **specialise** to
different curvatures (the learned κ vector spreading out)?


## 1. GPU


In [ ]:
!nvidia-smi -L || echo 'CPU (ok, small dims)'

## 2. Code + install (after a `git pull`, restart the runtime to pick up changes)


In [ ]:
import os
if not os.path.isdir('CurvLearn') and os.path.basename(os.getcwd()) != 'CurvLearn':
    !git clone https://github.com/TheFausap/CurvLearn.git
if os.path.basename(os.getcwd()) != 'CurvLearn':
    os.chdir('CurvLearn')
!pip -q install -e .
print('cwd:', os.getcwd())

## 3. Sanity: models (incl. product) forward/backward


In [ ]:
!python tests/test_model.py

## 4. Drive + output


In [ ]:
import os, torch
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTDIR = '/content/drive/MyDrive/CurvLearn/results'
except ModuleNotFoundError:
    OUTDIR = 'results'
os.makedirs(OUTDIR, exist_ok=True)
print('device:', dev, '| results ->', OUTDIR)

## 5. Run: shared vs per-head curvature
2 regimes x 2 factors x 3 seeds = 12 runs (small dims, fast), ~10-15 min. Checkpointed to
product_results.json.


In [ ]:
import json
from curvlearn.product import run_product_sweep, default_spec

spec = default_spec()
spec['train']['device'] = dev
ckpt = os.path.join(OUTDIR, 'product_results.json')
out = run_product_sweep(spec, checkpoint_path=ckpt, resume=True)
print('runs:', len(out['records']), '| saved ->', ckpt)

## 6. Plot: loss + learned per-head curvatures


In [ ]:
import json
from curvlearn import plot
from IPython.display import Image, display
out = json.load(open(os.path.join(OUTDIR, 'product_results.json')))
png = os.path.join(OUTDIR, 'product.png')
plot.product_plot(out, png)
display(Image(png))
print('saved', png)

## 7. Read it
- **per_head lower loss than shared, and the κ-per-head plot spreads** → mixed curvature helps
  and heads specialise (some more hyperbolic, some near-flat) — the product manifold earns its
  keep; natural next step is 𝕊×ℤ×ℍ with signed curvature (allow κ>0 spherical heads).
- **per_head ≈ shared, κ-per-head collapses to one value** → heads want the same curvature; a
  single κ suffices, mixing adds parameters without benefit here.
- **per_head worse** → independent per-head κ is harder to optimise at this scale; revisit
  kappa_lr or share κ across a few factors instead of every head.
